SEC fillings

1. Companies are required to file many financial reports with the SEC each year.
2. An Important form is the 10-K, which is an annual report of the companies activities.
3. These forms are public records, and can be accessed through the SEC's EDGAR database.

Data Cleaning

1. Completed 10-k forms are available to download as text files  that contain XML components
2. In order to work with them, the following cleaning steps were applied.
    1. Cleaned up the files using regex
    2. Parsed XML into python data structures using Beautiful soup
    3. Extracted CIK (Central Index Key) ID which is a company identifier used by SEC
    4. Extracted specific sections of the form (items 1, 1a,7 and 7a)
3. You can look in the data directory in the notebook if you did like to examine the cleaned data for yourself.


Plan of attack

1. Split form sections into chunks using a Langchain text splitter.
2. Create a graph where each chunk is a node, adding chunk metadata as properties.
3. Create a vector index.
4. Calculate the text embedding vector for each chunk and populate the index.
5. Use Similarity search to find relevant chunks


In [1]:
import os
from dotenv import load_dotenv

import json
import textwrap

# Langchain

from langchain_community.graphs import Neo4jGraph
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Neo4jVector
from langchain.chains import RetrievalQAWithSourcesChain
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings


In [2]:
load_dotenv()

#Environment variables
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

#GLobal Constants
VECTOR_INDEX_NAME = 'form_10l_chunks'  # single 10k documents 
VECTOR_NODE_LABEL = 'Chunk'
VECTOR_SOURCE_PROPERTY = 'text'
VECTOR_EMBEDDING_PROPERTY = 'textEmbedding'


In [3]:
first_file_name = "/Users/apple/Desktop/GenAI/GenAI/Knowledge_Graphs/Data/0000950170-23-027948.json"

In [4]:
first_file_as_object = json.load(open(first_file_name))

In [5]:
type(first_file_as_object)

dict

In [6]:
#will check for keys and type of values of object in dictionary


for k,v in first_file_as_object.items():
    print(k,type(v))

item1 <class 'str'>
item1a <class 'str'>
item7 <class 'str'>
item7a <class 'str'>
cik <class 'str'>
cusip6 <class 'str'>
cusip <class 'list'>
names <class 'list'>
source <class 'str'>


In [7]:
item1_text = first_file_as_object['item1']

In [8]:
item1_text[0:1500]

'>Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.\n\n\nOur opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of next-generation technologies, such as AI, Kubernetes, and modern databases. Our modern approach to hybrid, multicloud infrastructure and data management, which we term ‘evolved cloud’, provi

As we can see the text is large and hence we will perform chunking so that we are not going to take entire text and store that in a single record.

chunking is done using textsplitter from langchain.

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 2000,
    chunk_overlap = 200,
    length_function = len,
    is_separator_regex = False,
)

In [10]:
item1_text_chunks = text_splitter.split_text(item1_text)

In [11]:
type(item1_text_chunks)

list

In [12]:
len(item1_text_chunks)

254

In [13]:
item1_text_chunks[0]

'>Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.\n\n\nOur opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of next-generation technologies, such as AI, Kubernetes, and modern databases. Our modern approach to hybrid, multicloud infrastructure and data management, which we term ‘evolved cloud’, provi

In [14]:
# will create an helper function which will go through file and each section and create a chunk out of them.

def split_form10k_data_from_file(file):
    chunks_with_metadata = [] # use this to accumulate chunks records
    file_as_object = json.load(open(file)) # open json file
    
    for item in['item1','item1a','item7','item7a']: # pull these keys from the json
        print(f'Processing {item} from {file}')
        item_text = file_as_object[item] # grab the text of the item
        item_text_chunks = text_splitter.split_text(item_text) # split the text into chunks
        chunk_seq_id = 0 
        
        for chunk in item_text_chunks[:20]: # only take the first 20 chunks
            form_id = file[file.rindex('/') + 1:file.rindex('.')]  # Extract the form id file name.
            
            # finally, construct a record with metadata and the chunk text
            chunks_with_metadata.append({
                'text':chunk,
                # metadata from looping...
                'f10kItem': item,
                'chunkSeqId': chunk_seq_id,
                # construct metadata
                'formId': f'{form_id}', # pulled from the filename
                'chunkId': f'{form_id}-{item}-chunk{chunk_seq_id:04d}',
                #metadata from file...
                'names': file_as_object['names'],
                'cik': file_as_object['cik'],
                'cusip6': file_as_object['cusip6'],
                'source': file_as_object['source'],
            })
            chunk_seq_id +=1
        print(f'\tSplit into {chunk_seq_id} chunks')
    return chunks_with_metadata

In [15]:
first_file_chunks = split_form10k_data_from_file(first_file_name)

Processing item1 from /Users/apple/Desktop/GenAI/GenAI/Knowledge_Graphs/Data/0000950170-23-027948.json
	Split into 20 chunks
Processing item1a from /Users/apple/Desktop/GenAI/GenAI/Knowledge_Graphs/Data/0000950170-23-027948.json
	Split into 1 chunks
Processing item7 from /Users/apple/Desktop/GenAI/GenAI/Knowledge_Graphs/Data/0000950170-23-027948.json
	Split into 1 chunks
Processing item7a from /Users/apple/Desktop/GenAI/GenAI/Knowledge_Graphs/Data/0000950170-23-027948.json
	Split into 1 chunks


In [16]:
first_file_chunks[0]

{'text': '>Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.\n\n\nOur opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of next-generation technologies, such as AI, Kubernetes, and modern databases. Our modern approach to hybrid, multicloud infrastructure and data management, which we term ‘evolved clou

In [17]:
#use cypher query to merge the chunks into the graph

merge_chunk_node_query =""" 
MERGE(mergedChunk:Chunk {chunkID : $chunkParam.chunkId})
    ON CREATE SET
        mergedChunk.names = $chunkParam.names,
        mergedChunk.formId = $chunkParam.formId,
        mergedChunk.cik = $chunkParam.cik,
        mergedChunk.cusip6 = $chunkParam.cusip6,
        mergedChunk.source = $chunkParam.source,
        mergedChunk.f10Item = $chunkParam.f10kItem,
        mergedChunk.chunkSeqId = $chunkParam.chunkSeqId,
        mergedChunk.text = $chunkParam.text
RETURN mergedChunk
"""

In [18]:
kg = Neo4jGraph(
    url=NEO4J_URI, 
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD, 
    database=NEO4J_DATABASE
)

/var/folders/s8/qyjb36g92fs3ztdqk120mmkw0000gn/T/ipykernel_7157/1012656988.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  kg = Neo4jGraph(


In [19]:
kg.query(merge_chunk_node_query,
         params={'chunkParam':first_file_chunks[0]})

[{'mergedChunk': {'formId': '0000950170-23-027948',
   'names': ['Netapp Inc', 'NETAPP INC'],
   'cik': '1002047',
   'textEmbedding': [-0.017127543687820435,
    -0.018491918221116066,
    0.01617380604147911,
    -0.025936372578144073,
    -0.007126541808247566,
    0.013107272796332836,
    -0.004172603599727154,
    -0.012915200553834438,
    -0.003016858361661434,
    -0.03698383644223213,
    0.015961863100528717,
    0.013815953396260738,
    0.01384244579821825,
    -0.013378823176026344,
    0.0026111886836588383,
    0.004589863587170839,
    0.015087603591382504,
    -0.029274456202983856,
    -0.014981633052229881,
    -0.021512089297175407,
    -0.031340885907411575,
    0.010656696744263172,
    0.00014157047553453594,
    -0.006437730975449085,
    -0.015445255674421787,
    -0.010742797516286373,
    0.00411630654707551,
    -0.015445255674421787,
    0.01945890299975872,
    -0.004550124518573284,
    0.008504163473844528,
    -0.02006823569536209,
    -0.0079080769792

We should make sure we don't duplicate the data

In [20]:
kg.query("""
         CREATE CONSTRAINT unique_chunk IF NOT EXISTS
         FOR (c:Chunk) REQUIRE c.chunkId IS UNIQUE
         """)

[]

In [21]:
kg.query("SHOW INDEXES")

[{'id': 4,
  'name': 'form_10l_chunks',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'VECTOR',
  'entityType': 'NODE',
  'labelsOrTypes': ['Chunk'],
  'properties': ['textEmbedding'],
  'indexProvider': 'vector-2.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 0,
  'name': 'index_343aff4e',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'NODE',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': neo4j.time.DateTime(2025, 6, 19, 5, 51, 52, 234000000, tzinfo=<UTC>),
  'readCount': 65},
 {'id': 1,
  'name': 'index_f7700477',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'RELATIONSHIP',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 2,
  'name': 'unique_chunk',
  'state': 'ONLI

In [22]:
node_count=0
for chunk in first_file_chunks:
    print(f"Creating `:Chunk` node for chunk ID {chunk['chunkId']}")
    kg.query(merge_chunk_node_query,
             params={
                 'chunkParam':chunk
             })
    node_count += 1
print(f"Created {node_count} nodes")

Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0000
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0001
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0002
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0003
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0004
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0005
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0006
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0007
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0008
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0009
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0010
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0011
Creating `:Chunk` node for chunk ID 0000950170-23-027948-item1-chunk0012
Creating `:Chunk` node for chunk ID 0000950170-23-0

In [23]:
kg.query("""
         MATCH (n)
         RETURN count(n) as nodeCount
         """)

[{'nodeCount': 23}]

In [24]:
kg.query("""
         CREATE VECTOR INDEX `form_10l_chunks` IF NOT EXISTS
         FOR (c:Chunk) ON (c.textEmbedding)
         OPTIONS {
             indexConfig: {
                 `vector.dimensions` : 1536,
                 `vector.similarity_function`: 'cosine'
             }
         }
         """)

[]

In [25]:
kg.query("SHOW INDEXES")

[{'id': 4,
  'name': 'form_10l_chunks',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'VECTOR',
  'entityType': 'NODE',
  'labelsOrTypes': ['Chunk'],
  'properties': ['textEmbedding'],
  'indexProvider': 'vector-2.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 0,
  'name': 'index_343aff4e',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'NODE',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': neo4j.time.DateTime(2025, 6, 19, 5, 58, 17, 521000000, tzinfo=<UTC>),
  'readCount': 70},
 {'id': 1,
  'name': 'index_f7700477',
  'state': 'ONLINE',
  'populationPercent': 100.0,
  'type': 'LOOKUP',
  'entityType': 'RELATIONSHIP',
  'labelsOrTypes': None,
  'properties': None,
  'indexProvider': 'token-lookup-1.0',
  'owningConstraint': None,
  'lastRead': None,
  'readCount': 0},
 {'id': 2,
  'name': 'unique_chunk',
  'state': 'ONLI

In [26]:
kg.query("""
         MATCH (chunk:Chunk) WHERE chunk.textEmbedding IS NULL
         WITH chunk, genai.vector.encode(chunk.text,"OpenAI", {token:$openAiApiKey})AS vector
         CALL db.create.setNodeVectorProperty(chunk, "textEmbedding", vector)
         """,
         params={"openAiApiKey":OPENAI_API_KEY})

[]

In [27]:
kg.refresh_schema()

In [28]:
print(kg.schema)

Node properties:
Chunk {chunkID: STRING, names: LIST, formId: STRING, cik: STRING, cusip6: STRING, source: STRING, f10Item: STRING, chunkSeqId: INTEGER, text: STRING, textEmbedding: LIST}
Relationship properties:

The relationships:



In [29]:
def neo4j_vector_search(question):
    """Search for similar nodes using the Neo4j Vector Index"""
    vector_search_query = """ 
        WITH genai.vector.encode($question, "OpenAI", {token:$openAiApiKey}) AS question_embedding
        CALL db.index.vector.queryNodes($index_name, $top_k, question_embedding) yield node, score
        RETURN score, node.text AS text
    """
    
    similar = kg.query( vector_search_query,
                       params={
                           'question':question,
                           'openAiApiKey': OPENAI_API_KEY,
                           'index_name': VECTOR_INDEX_NAME,
                           'top_k':10
                       })
    return similar

In [30]:
search_results = neo4j_vector_search(
    'In a single sentence, tell me about Netaapp.'
)

In [31]:
search_results[0]

{'score': 0.900238037109375,
 'text': '>Item 1.  \nBusiness\n\n\nOverview\n\n\nNetApp, Inc. (NetApp, we, us or the Company) is a global cloud-led, data-centric software company. We were incorporated in 1992 and are headquartered in San Jose, California. Building on more than three decades of innovation, we give customers the freedom to manage applications and data across hybrid multicloud environments. Our portfolio of cloud services, and storage infrastructure, powered by intelligent data management software, enables applications to run faster, more reliably, and more securely, all at a lower cost.\n\n\nOur opportunity is defined by the durable megatrends of data-driven digital and cloud transformations. NetApp helps organizations meet the complexities created by rapid data and cloud growth, multi-cloud management, and the adoption of next-generation technologies, such as AI, Kubernetes, and modern databases. Our modern approach to hybrid, multicloud infrastructure and data management

In [33]:
neo4j_vector_store = Neo4jVector.from_existing_graph(
    embedding=OpenAIEmbeddings(),
    url=NEO4J_URI, 
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD, 
    database=NEO4J_DATABASE,
    index_name=VECTOR_INDEX_NAME,
    node_label=VECTOR_NODE_LABEL,
    text_node_properties=[VECTOR_SOURCE_PROPERTY],
    embedding_node_property=VECTOR_EMBEDDING_PROPERTY,
)

In [34]:
retriever = neo4j_vector_store.as_retriever()

In [ ]:
chain = RetrievalQAWithSourcesChain.from_chain_type(
    ChatOpenAI(temperature=0),
    chain_type = "stuff",  # combines all into 1 prompt and then pass it to LLM
)

In [47]:
def prettychain(question:str) -> str:
    """ Pretty print the chain's response to a question"""
    response = chain({"question":question},
                    return_only_outputs=True,)
    print(textwrap.fill(response['answer'],80))

In [48]:
question = "What is Netapp's primary business?"

In [49]:
prettychain(question)

/var/folders/s8/qyjb36g92fs3ztdqk120mmkw0000gn/T/ipykernel_7157/2928577645.py:3: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = chain({"question":question},


NetApp's primary business is enterprise storage and data management, cloud
storage, and cloud operations.


In [50]:
prettychain("Where is headquarter of Netapp?")

Netapp is headquartered in San Jose, California.


In [51]:
prettychain("""
            Tell me about Netapp.
            Limit your answer to a single sentence.
            """)

NetApp is a global cloud-led, data-centric software company that provides
customers with the freedom to manage applications and data across hybrid
multicloud environments.


In [52]:
prettychain("""
            Tell me about Apple.
            Limit your answer to single sentence.
            """)

Apple is a global cloud-led, data-centric software company headquartered in San
Jose, California.


This is classic hallucination as its giving details of netapp but with apple name.

In [ ]:
prettychain("""
            Tell me about Apple.
            Limit your answer to single sentence.
            If you are unsure about the answer, say you don't know
            """)

I don't know.


Here we used neo4j as vector store but not really as knowledge graph.